# 02 — YOLOv12 Training
**Drone Detection | ITMO Diploma Thesis**

Обучаем две версии YOLOv12:
- `yolo12n` (nano) — максимальная скорость, реальное время
- `yolo12s` (small) — баланс точность/скорость

Метрики: mAP@0.5, mAP@0.5:0.95, Precision, Recall, FPS

**I/O:** лейблы — отдельной ячейкой (распаковка zip в `/content/drone_yolo_local`). **CELL 3** копирует с Drive только **`prepared/yolo/images`** → быстрый локальный диск.

In [ ]:
# ── CELL 1: Install & check GPU ────────────────────────────────────────────────
!uv pip install ultralytics
import ultralytics; ultralytics.checks()
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No GPU detected! Switch runtime to GPU in Runtime → Change runtime type'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# ── CELL 2: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL 3: только images с Drive (лейблы — ячейкой выше из zip) ─────────────
import os
import time
import json
import shutil
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import pandas as pd
from ultralytics import YOLO

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks')
DRIVE_DATA = DRIVE_ROOT / 'prepared' / 'yolo'
WEIGHTS_DIR = DRIVE_ROOT / 'weights'
RESULTS_DIR = DRIVE_ROOT / 'results'

LOCAL_DATA = Path('/content/drone_yolo_local')
LOCAL_IMAGES = LOCAL_DATA / 'images'
DRIVE_IMAGES = DRIVE_DATA / 'images'
FORCE_RECOPY_IMAGES = False  # True: удалить локальный images/ и скопировать с Drive снова

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert DRIVE_IMAGES.is_dir(), f'На Drive нет папки: {DRIVE_IMAGES}'

LOCAL_DATA.mkdir(parents=True, exist_ok=True)

if LOCAL_IMAGES.exists() and FORCE_RECOPY_IMAGES:
    shutil.rmtree(LOCAL_IMAGES)

if not LOCAL_IMAGES.exists():
    print('Копирование images: Drive → /content/drone_yolo_local/images …')
    t0 = time.perf_counter()
    shutil.copytree(DRIVE_IMAGES, LOCAL_IMAGES)
    print(f'Готово за {time.perf_counter() - t0:.1f} с')
else:
    print('images уже есть:', LOCAL_IMAGES, '(FORCE_RECOPY_IMAGES=True — перекопировать)')

DATA_DIR = LOCAL_DATA
YAML_PATH = DATA_DIR / 'data.yaml'

_root = DATA_DIR.resolve()
_imgs = _root / 'images'

def _split(name: str) -> str | None:
    p = _imgs / name
    return f'images/{name}' if p.is_dir() else None

_train = _split('train')
_val = _split('val')
_test = _split('test')

if _train is None:
    _listing = sorted(p.name for p in _imgs.iterdir()) if _imgs.is_dir() else []
    raise FileNotFoundError(
        f'Нет images/train в {_imgs}. Содержимое images/: {_listing}'
    )
if _val is None:
    _listing = sorted(p.name for p in _imgs.iterdir()) if _imgs.is_dir() else []
    raise FileNotFoundError(
        f'Нет images/val в {_imgs}. Содержимое images/: {_listing}'
    )
if _test is None:
    _test = _val
    print('Предупреждение: images/test нет — test в data.yaml = val (для train ок; для финального теста добавьте split).')

_yaml_fixed = f'''# Local Colab — fast I/O (dataset copied from Drive in this notebook)
path: {_root.as_posix()}
train: {_train}
val: {_val}
test: {_test}
nc: 4
names: ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
'''
YAML_PATH.write_text(_yaml_fixed, encoding='utf-8')
for _rel in (_train, _val, _test):
    _p = _root / _rel
    assert _p.is_dir(), f'После записи data.yaml нет папки: {_p}'
print(f'data.yaml → path = {_root.as_posix()} ✓ (train={_train}, val={_val}, test={_test})')

_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3) if torch.cuda.is_available() else 0
print(f'GPU VRAM ~{_vram_gb:.1f} GB — при OOM уменьшите batch в TRAIN_CONFIG')

## Конфигурация обучения

In [ ]:
# ── CELL 4: Training config ───────────────────────────────────────────────────
TRAIN_CONFIG = {
    'data':       str(YAML_PATH),
    'epochs':     50,
    'imgsz':      640,
    'batch':      16,          # T4: 16; после копии на /content можно 24–32 на 16+ GB VRAM
    'device':     0,           # GPU 0
    'workers':    8,          # больше workers при быстром локальном диске
    'cache':      'disk',      # кеш датасета на диск (ускоряет повторные эпохи); при ошибке уберите или поставьте False
    'patience':   10,          # early stopping
    'save':       True,
    'pretrained': True,        # transfer learning from COCO
    'optimizer':  'AdamW',
    'lr0':        0.001,
    'lrf':        0.01,
    'momentum':   0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    # Augmentation
    'hsv_h':      0.015,
    'hsv_s':      0.7,
    'hsv_v':      0.4,
    'degrees':    10.0,
    'translate':  0.1,
    'scale':      0.5,
    'fliplr':     0.5,
    'mosaic':     1.0,
    'mixup':      0.1,
    'copy_paste': 0.1,
}
print('Training config ready.')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')

## 1. Обучение YOLOv12n (nano)

In [ ]:
# ── CELL 5: Train YOLOv12n ────────────────────────────────────────────────────
print('=' * 60)
print('Training YOLOv12n (nano)')
print('=' * 60)

model_n = YOLO('yolo12n.pt')  # auto-downloads pretrained weights

results_n = model_n.train(
    name='yolo12n_drone',
    project='/content/runs',
    **TRAIN_CONFIG
)

# save_dir с суффиксом при повторных запусках (yolo12n_drone2, …)
best_n_src = Path(model_n.trainer.save_dir) / 'weights' / 'best.pt'
best_n_dst = WEIGHTS_DIR / 'yolo12n_drone_best.pt'
assert best_n_src.is_file(), f'Нет {best_n_src}'
shutil.copy(best_n_src, best_n_dst)
print(f'\nBest weights saved → {best_n_dst} (from {best_n_src.parent.parent.name})')

## 2. Обучение YOLOv12s (small)

In [ ]:
# ── CELL 6: Train YOLOv12s ────────────────────────────────────────────────────
print('=' * 60)
print('Training YOLOv12s (small)')
print('=' * 60)

model_s = YOLO('yolo12s.pt')

results_s = model_s.train(
    name='yolo12s_drone',
    project='/content/runs',
    **TRAIN_CONFIG
)

best_s_src = Path(model_s.trainer.save_dir) / 'weights' / 'best.pt'
best_s_dst = WEIGHTS_DIR / 'yolo12s_drone_best.pt'
assert best_s_src.is_file(), f'Нет {best_s_src}'
shutil.copy(best_s_src, best_s_dst)
print(f'\nBest weights saved → {best_s_dst} (from {best_s_src.parent.parent.name})')

## 3. Валидация и метрики

In [ ]:
# ── CELL 7: Validate both models on test set ──────────────────────────────────
def validate_yolo(weights_path: Path, data_yaml: Path,
                  split: str = 'test') -> dict:
    """Run YOLO validation and return metrics dict.

    Args:
        weights_path: Path to .pt weights file.
        data_yaml: Path to data.yaml.
        split: Dataset split to evaluate on.

    Returns:
        Dict with mAP50, mAP50_95, precision, recall keys.
    """
    model = YOLO(str(weights_path))
    metrics = model.val(data=str(data_yaml), split=split, device=0, verbose=True)
    return {
        'mAP50':      round(float(metrics.box.map50),    4),
        'mAP50_95':   round(float(metrics.box.map),      4),
        'precision':  round(float(metrics.box.mp),       4),
        'recall':     round(float(metrics.box.mr),       4),
        'per_class_mAP50': metrics.box.ap50.tolist(),
    }


metrics_n = validate_yolo(WEIGHTS_DIR / 'yolo12n_drone_best.pt', YAML_PATH)
metrics_s = validate_yolo(WEIGHTS_DIR / 'yolo12s_drone_best.pt', YAML_PATH)

print('\n─── YOLOv12n ───')
for k, v in metrics_n.items():
    print(f'  {k}: {v}')
print('\n─── YOLOv12s ───')
for k, v in metrics_s.items():
    print(f'  {k}: {v}')

In [ ]:
# ── CELL 8: Measure FPS ───────────────────────────────────────────────────────
def measure_fps(weights_path: Path, test_img_dir: Path,
                n_runs: int = 100) -> float:
    """Measure average inference FPS on GPU.

    Args:
        weights_path: Path to model weights.
        test_img_dir: Directory of test images.
        n_runs: Number of inference runs for averaging.

    Returns:
        Average FPS.
    """
    import glob
    model = YOLO(str(weights_path))
    imgs = (list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))[:n_runs]
    if not imgs:
        return 0.0

    # Warmup
    for _ in range(5):
        model.predict(str(imgs[0]), device=0, verbose=False)

    start = time.perf_counter()
    for img in imgs:
        model.predict(str(img), device=0, verbose=False)
    elapsed = time.perf_counter() - start
    return round(len(imgs) / elapsed, 1)


test_img_dir = DATA_DIR / 'images' / 'test'
fps_n = measure_fps(WEIGHTS_DIR / 'yolo12n_drone_best.pt', test_img_dir)
fps_s = measure_fps(WEIGHTS_DIR / 'yolo12s_drone_best.pt', test_img_dir)
print(f'YOLOv12n FPS: {fps_n}')
print(f'YOLOv12s FPS: {fps_s}')

In [ ]:
# ── CELL 9: Save YOLO metrics to JSON (merge; keeps yolo26n/s if present) ───
CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']

out = RESULTS_DIR / 'yolo_metrics.json'
existing = {}
if out.exists():
    with open(out) as f:
        existing = json.load(f)

existing['yolo12n'] = {**metrics_n, 'fps': fps_n, 'size_mb': round(
    (WEIGHTS_DIR / 'yolo12n_drone_best.pt').stat().st_size / 1e6, 1)}
existing['yolo12s'] = {**metrics_s, 'fps': fps_s, 'size_mb': round(
    (WEIGHTS_DIR / 'yolo12s_drone_best.pt').stat().st_size / 1e6, 1)}
existing['class_names'] = CLASS_NAMES

with open(out, 'w') as f:
    json.dump(existing, f, indent=2)
print(f'Metrics saved → {out}')

In [ ]:
# ── CELL 10: Plot training curves ─────────────────────────────────────────────
def plot_training_curves(run_dir: str, model_name: str) -> None:
    """Plot loss and mAP curves from YOLO results.csv.

    Args:
        run_dir: Path to training run directory.
        model_name: Model name for plot title.
    """
    csv_path = Path(run_dir) / 'results.csv'
    if not csv_path.exists():
        print(f'results.csv not found: {csv_path}')
        return
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'{model_name} Training Curves', fontsize=14, fontweight='bold')

    plots = [
        ('train/box_loss',  'Train Box Loss',     axes[0, 0]),
        ('train/cls_loss',  'Train Class Loss',   axes[0, 1]),
        ('train/dfl_loss',  'Train DFL Loss',     axes[0, 2]),
        ('metrics/mAP50(B)', 'mAP@0.5',          axes[1, 0]),
        ('metrics/mAP50-95(B)', 'mAP@0.5:0.95',  axes[1, 1]),
        ('metrics/precision(B)', 'Precision',     axes[1, 2]),
    ]

    for col, title, ax in plots:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color='#3498db', linewidth=2)
            ax.set_title(title)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
        else:
            ax.set_title(f'{title} (N/A)')

    plt.tight_layout()
    save_path = RESULTS_DIR / f'{model_name}_curves.png'
    plt.savefig(save_path, dpi=150)
    print(f'Curves saved → {save_path}')
    plt.show()


def _latest_yolo_run(prefix: str) -> str:
    """Последний по времени run (yolo12n_drone7 и т.д.)."""
    root = Path('/content/runs')
    matches = sorted(root.glob(f'{prefix}*'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f'Нет каталога {root}/{prefix}*')
    return str(matches[0])


plot_training_curves(_latest_yolo_run('yolo12n_drone'), 'YOLOv12n')
plot_training_curves(_latest_yolo_run('yolo12s_drone'), 'YOLOv12s')

In [ ]:
# ── CELL 11: Export models to ONNX ────────────────────────────────────────────
# Ultralytics сохраняет .onnx рядом с .pt (то же имя базы) — копировать не нужно.
for model_name, weights in [('yolo12n', WEIGHTS_DIR / 'yolo12n_drone_best.pt'),
                              ('yolo12s', WEIGHTS_DIR / 'yolo12s_drone_best.pt')]:
    model = YOLO(str(weights))
    model.export(format='onnx', imgsz=640, simplify=True)
    onnx_path = weights.with_suffix('.onnx')
    if onnx_path.exists():
        print(f'{model_name} ONNX → {onnx_path}')
    else:
        print(f'Warning: ONNX not found after export: {onnx_path}')

In [ ]:
# ── CELL 12: Quick prediction visualization ───────────────────────────────────
import random
from PIL import Image
import numpy as np

model_s_loaded = YOLO(str(WEIGHTS_DIR / 'yolo12s_drone_best.pt'))
test_imgs = list((DATA_DIR / 'images' / 'test').glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
fig.suptitle('YOLOv12s Predictions on Test Set', fontsize=13, fontweight='bold')

for ax, img_path in zip(axes, test_imgs):
    results = model_s_loaded.predict(str(img_path), device=0, conf=0.25, verbose=False)
    annotated = results[0].plot()
    ax.imshow(annotated[..., ::-1])  # BGR→RGB
    ax.axis('off')
    ax.set_title(img_path.name[:25], fontsize=8)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'yolo12s_predictions.png', dpi=120)
plt.show()
print('Done! Next step → 03_faster_rcnn_train.ipynb')

In [ ]:
# ── CELL 13: Переоценка YOLOv12n / YOLOv12s на текущем data.yaml → yolo_metrics.json ─
# Нужны: CELL 1–3 (YAML_PATH, WEIGHTS_DIR, RESULTS_DIR, DATA_DIR), файлы весов v12 на Drive.

VAL_SPLIT = 'val'  # 'val' или 'test' — какой сплит из data.yaml оценивать
RERUN_FPS = False  # True — пересчитать FPS (как CELL 8); False — взять fps из существующего JSON


def validate_yolo_v12(weights_path: Path, data_yaml: Path, split: str) -> dict:
    """Метрики детекции на указанном сплите (тот же формат, что CELL 7)."""
    model = YOLO(str(weights_path))
    metrics = model.val(data=str(data_yaml), split=split, device=0, verbose=True)
    return {
        'mAP50': round(float(metrics.box.map50), 4),
        'mAP50_95': round(float(metrics.box.map), 4),
        'precision': round(float(metrics.box.mp), 4),
        'recall': round(float(metrics.box.mr), 4),
        'per_class_mAP50': metrics.box.ap50.tolist(),
    }


def measure_fps_v12(weights_path: Path, test_img_dir: Path, n_runs: int = 100) -> float:
    model = YOLO(str(weights_path))
    imgs = (list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png')))[:n_runs]
    if not imgs:
        return 0.0
    for _ in range(5):
        model.predict(str(imgs[0]), device=0, verbose=False)
    start = time.perf_counter()
    for img in imgs:
        model.predict(str(img), device=0, verbose=False)
    elapsed = time.perf_counter() - start
    return round(len(imgs) / elapsed, 1)


w_n = WEIGHTS_DIR / 'yolo12n_drone_best.pt'
w_s = WEIGHTS_DIR / 'yolo12s_drone_best.pt'
assert w_n.exists(), f'Нет весов: {w_n}'
assert w_s.exists(), f'Нет весов: {w_s}'

metrics_n = validate_yolo_v12(w_n, YAML_PATH, VAL_SPLIT)
metrics_s = validate_yolo_v12(w_s, YAML_PATH, VAL_SPLIT)

print(f'─── YOLOv12n (split={VAL_SPLIT!r}) ───')
for k, v in metrics_n.items():
    print(f'  {k}: {v}')
print(f'─── YOLOv12s (split={VAL_SPLIT!r}) ───')
for k, v in metrics_s.items():
    print(f'  {k}: {v}')

out = RESULTS_DIR / 'yolo_metrics.json'
existing: dict = {}
if out.exists():
    with open(out, encoding='utf-8') as f:
        existing = json.load(f)

test_img_dir = DATA_DIR / 'images' / 'test'
if RERUN_FPS:
    fps_n = measure_fps_v12(w_n, test_img_dir)
    fps_s = measure_fps_v12(w_s, test_img_dir)
    print(f'FPS (пересчёт): yolo12n={fps_n}, yolo12s={fps_s}')
else:
    prev_n = existing.get('yolo12n') or {}
    prev_s = existing.get('yolo12s') or {}
    fps_n = prev_n.get('fps')
    fps_s = prev_s.get('fps')
    if fps_n is None or fps_s is None:
        raise KeyError('В JSON нет fps для yolo12n/yolo12s — выполните с RERUN_FPS = True')
    print(f'FPS (из сохранённого JSON): yolo12n={fps_n}, yolo12s={fps_s}')

CLASS_NAMES = ['DRONE', 'AIRPLANE', 'HELICOPTER', 'BIRD']
existing['yolo12n'] = {
    **metrics_n,
    'fps': fps_n,
    'size_mb': round(w_n.stat().st_size / 1e6, 1),
}
existing['yolo12s'] = {
    **metrics_s,
    'fps': fps_s,
    'size_mb': round(w_s.stat().st_size / 1e6, 1),
}
existing['class_names'] = CLASS_NAMES

with open(out, 'w', encoding='utf-8') as f:
    json.dump(existing, f, indent=2)
print(f'Обновлены yolo12n / yolo12s → {out}')